# ICT4403 Assignment 3 — English-to-French translation
**Mayaank Kothari · 230953562**

This notebook trains a scratch Transformer and mT5, then evaluates prompted LLaMA and Gemma on the same saved test split. It records actual scores and timings. Choose **Runtime → Change runtime type → GPU**. A T4-class GPU is a practical starting point; runtime availability and duration vary. Models are loaded sequentially.

The saved repository may contain partial CPU results. Run every model here for a consistent hardware comparison. Do not claim missing runs are complete.


In [ ]:
from pathlib import Path
import subprocess, os
work = Path('/content/Fgai_Assgn_2')
if not work.exists():
    subprocess.run(['git','clone','https://github.com/KothariMayaank/Fgai_Assgn_2.git',str(work)],check=True)
os.chdir(work/'assignment3')
print(Path.cwd())


In [ ]:
%pip install -q -r requirements.txt
import torch
print('PyTorch:',torch.__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
assert torch.cuda.is_available(), 'Select a GPU runtime before the full experiment.'


## Model access
In your own Hugging Face account, review and accept access conditions for:
- https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
- https://huggingface.co/google/gemma-3-1b-it

Use the login widget below, or configure an `HF_TOKEN` Colab secret. Never put a token in a code cell, chat, repository or downloaded report. Authentication alone does not grant gated-model access. Approval for LLaMA may require waiting.


In [ ]:
from huggingface_hub import notebook_login, hf_hub_download
notebook_login()
for model in ['google/mt5-small','meta-llama/Llama-3.2-1B-Instruct','google/gemma-3-1b-it']:
    hf_hub_download(model,'config.json')
    print('Access verified:',model)


## Verify the shared data
Use the committed 1,024 / 128 / 128 split for this experiment. This is a deliberately small resource-limited subset. For stronger conclusions, increase all split sizes and rerun every model. Do not tune on test scores. Keep corpus attribution and manifest hashes.


In [ ]:
import json
print(json.dumps(json.load(open('data/manifest.json')),indent=2))
subprocess.run(['python','-m','unittest','test_pipeline','-v'],check=True)


## Train the encoder–decoder systems
The lowest validation-loss checkpoint is selected. Training and validation time is recorded separately from inference. mT5 requires more memory and runtime than the scratch Transformer.


In [ ]:
subprocess.run(['python','experiment.py','basic','--epochs','15'],check=True)


In [ ]:
subprocess.run(['python','experiment.py','mt5','--epochs','2'],check=True)


## Prompt the decoder-only systems
Both receive the same three training demonstrations. Primary scores use greedy generation; optional sweeps evaluate temperatures 0.3 / 0.7 / 1.0 with three seeds each. Sweeps require substantially more inference than the main test.


In [ ]:
subprocess.run(['python','experiment.py','llama','--temperatures'],check=True)


In [ ]:
subprocess.run(['python','experiment.py','gemma','--temperatures'],check=True)


## Summarize and review
Inspect all translations, not just headline scores. Review diagnostic probes for gender bias, hallucinations, cultural meaning and sensitive content. A bilingual reviewer should fill in the human-review notes; the notebook does not claim an automated safety pass.


In [ ]:
subprocess.run(['python','experiment.py','summarize'],check=True)
from IPython.display import display, Markdown, Image
display(Markdown(Path('results/RESULTS.md').read_text()))
display(Image(filename='results/quality.png'))
for model in ['basic','mt5','llama','gemma']:
    predictions=json.load(open(f'results/{model}_translations.json'))
    print(model, predictions[:3])


## Generate submission documents
The cover contains the student details, dataset link and GitHub folder link. The report reads the actual saved results. Export DOCX files using LibreOffice below or Word's Save as PDF. Check every PDF page before uploading to LMS.


In [ ]:
subprocess.run(['python','review_translations.py'],check=True)
subprocess.run(['python','make_documents.py'],check=True)


In [ ]:
# Install the document converter in this Colab runtime.
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','libreoffice-writer'],check=True)
subprocess.run(['libreoffice','-env:UserInstallation=file:///tmp/fgai-lo','--headless','--convert-to','pdf',
                'Mayaank_Kothari_230953562_FGAI_Assignment3.docx','Assignment3_Report.docx','--outdir','.'],check=True)


## Save your work
The ZIP below includes source, exact splits, results and documents. It excludes downloaded model weights and secrets. Upload the updated files to the assignment3 GitHub folder, ensure the evaluator can access them, then upload the cover PDF to LMS. Nothing is submitted to the college automatically.


In [ ]:
import zipfile
out=Path('/content/Mayaank_Kothari_Assignment3.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for path in Path('.').rglob('*'):
        if path.is_file() and not any(x in path.parts for x in ['checkpoints','raw','__pycache__','.venv','.git']):
            if path.suffix!='.log' and path.name!='.env': z.write(path,str(Path('assignment3')/path))
from google.colab import files
files.download(str(out))
